# GLACE — exploration interactive
Modifie les paramètres dans la cellule **Paramètres**, puis exécute **Run All** (ou Shift+Entrée cellule par cellule).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from types import SimpleNamespace
from glace.main import build_params
from glace.core import run_simulation
from glace.figures import plot_profiles, plot_snapshots, plot_attribution, plot_data_vs_model
print('✓ imports OK')

## Paramètres
Change ce que tu veux ici, puis Shift+Entrée sur toutes les cellules suivantes.

In [ ]:
# ── Physique du gaz ───────────────────────────────────────────────────────────
P_diss  = 5e6        # pression de dissociation des clathrates   [Pa]
S_Ar    = 10e-9      # solubilité Ar dans le réseau              [mol m⁻³ Pa⁻¹]

# ── Site ─────────────────────────────────────────────────────────────────────
TAC     = 4.856e-5   # total air content (données labo)          [m³_STP kg⁻¹]
b_acc   = 0.35       # taux d'accumulation                       [m yr⁻¹]
H_ice   = 1387.0     # épaisseur de glace                        [m]
p_llib  = 3          # paramètre de forme Lliboutry              [-]

# ── Transport ─────────────────────────────────────────────────────────────────
D_m2s       = 1e-11  # diffusivité dans le réseau                [m² s⁻¹]
flux_bottom = 6e-7   # flux basal Ar                             [mol m⁻² yr⁻¹]

# ── Gas loss ──────────────────────────────────────────────────────────────────
GAS_LOSS_MODE = 'none'   # 'none' ou 'data'
gas_loss_data = 1.8e-6   # taux max (mode 'data' seulement)      [yr⁻¹]
z_char        = 8.0      # hauteur caractéristique               [m]
z_gl_limit    = 20.0     # limite supérieure de la zone          [m]

# ── Grille et temps ───────────────────────────────────────────────────────────
nz = 101
Z  = 30.0            # domaine au-dessus du bed                  [m]
T  = 1e5             # durée totale de simulation                [yr]
dt = 3.0             # pas de temps                              [yr]

# ── Constantes ────────────────────────────────────────────────────────────────
rho_ice = 917.0
Vm_STP  = 22.414e-3
X_atm   = 0.934e-2   # référence atmosphérique (non utilisé pour BC/IC)

## Lancement de la simulation

In [ ]:
p = SimpleNamespace(
    P_diss=P_diss, S_Ar=S_Ar, TAC=TAC, b_acc=b_acc, H_ice=H_ice, p_llib=p_llib,
    D_m2s=D_m2s, flux_bottom=flux_bottom,
    GAS_LOSS_MODE=GAS_LOSS_MODE, gas_loss_data=gas_loss_data,
    z_char=z_char, z_gl_limit=z_gl_limit,
    nz=nz, Z=Z, T=T, dt=dt,
    rho_ice=rho_ice, Vm_STP=Vm_STP, X_atm=X_atm,
)
sim = build_params(p)
result = run_simulation(sim, return_full=True)
print(f'Simulation terminée — {sim["nt"]} pas de temps')

## Figures

In [ ]:
fig1 = plot_profiles(result, title=f'Profils finaux  (T = {T:.2g} yr)')
plt.show()

In [ ]:
fig2 = plot_snapshots(result, n_snap=6, title=f'Évolution temporelle  (T = {T:.2g} yr)')
plt.show()

In [ ]:
fig5 = plot_data_vs_model(result, title=f'Modèle vs observations  (T = {T:.2g} yr)')
plt.show()

## Attribution des processus
Compare : diffusion seule → + advection → + gas loss

In [ ]:
z = sim['z']

p1 = dict(sim); p1['u'] = np.zeros_like(sim['u']); p1['lambda_z'] = np.zeros_like(sim['lambda_z'])
p2 = dict(sim); p2['lambda_z'] = np.zeros_like(sim['lambda_z'])
p3 = dict(sim)

r1 = run_simulation(p1)
r2 = run_simulation(p2)
r3 = run_simulation(p3)

fig4 = plot_attribution(
    {'Diffusion + flux basal': r1['C_bulk'], '+ Advection': r2['C_bulk'], '+ Gas loss (complet)': r3['C_bulk']},
    z, title=f'Attribution des processus  (T = {T:.2g} yr)'
)
plt.show()